# Checkpoint 07 — desafio final de processamento

Este checkpoint **continua a estrutura dos checkpoints anteriores**.

Não criamos um novo cliente SSE. Reutilizamos:

- `firebase_node_url()`;
- `iterar_sse()`;
- `pacotes_firebase()`;
- `validar_pacote()`.

A novidade é o **pós-processamento** de uma janela contínua:

1. filtragem passa-faixa;
2. detecção de picos R no ECG;
3. detecção de pulsos no PPG;
4. estimativa didática de frequência cardíaca e frequência de pulso.

> Os parâmetros são didáticos. Este notebook não implementa um algoritmo clínico validado e não produz diagnóstico.


## 1. Configuração e bibliotecas

Para este desafio, deixe o gerador inicialmente em **`NORMAL`**.
Depois, como bônus, você pode repetir com `NOISE` ou `MOTION`.


In [ ]:
import json                         # Converte as mensagens JSON recebidas pelo SSE.
import time                         # Controla a duração da aquisição.
import matplotlib.pyplot as plt     # Mostra os sinais filtrados e os picos detectados.
import numpy as np                  # Faz os cálculos numéricos.
import requests                     # Recebe os pacotes do Firebase por HTTP/SSE.
from scipy.signal import butter, sosfiltfilt, find_peaks  # Filtra os sinais e detecta picos.

FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Mesmo banco usado nos checkpoints anteriores.
STREAM_PATH = "oficina/live"  # Mesmo nó usado durante toda a oficina.
DURACAO_SEGUNDOS = 20         # Janela de aquisição usada no desafio.


## 2. Reutilização do cliente dos checkpoints anteriores

As funções abaixo mantêm os mesmos nomes e a mesma lógica do Checkpoint 06.


In [ ]:
# Função já utilizada anteriormente: monta a URL REST do nó do Firebase.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//".
    node = node.strip("/")           # Remove barras extras do caminho do nó.
    return f"{base}/{node}.json"     # A API REST do Realtime Database usa o sufixo .json.


# Função já utilizada anteriormente: separa a resposta SSE em eventos completos.
def iterar_sse(response):
    """Interpreta os eventos Server-Sent Events retornados pelo Firebase REST."""
    evento = None
    dados = []

    for raw in response.iter_lines(decode_unicode=True):
        if raw is None:
            continue

        linha = raw.strip()

        if linha == "":
            if evento is not None and dados:
                yield evento, "\n".join(dados)
            evento = None
            dados = []
            continue

        if linha.startswith(":"):
            continue

        if linha.startswith("event:"):
            evento = linha.split(":", 1)[1].strip()
        elif linha.startswith("data:"):
            dados.append(linha.split(":", 1)[1].lstrip())


# Função já utilizada anteriormente: recebe, um a um, os pacotes publicados no Firebase.
def pacotes_firebase(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=20):
    url = firebase_node_url(firebase_url, node)
    headers = {"Accept": "text/event-stream"}
    inicio = time.time()

    with requests.get(
        url,
        headers=headers,
        stream=True,
        allow_redirects=True,
        timeout=(10, 35),
    ) as response:
        response.raise_for_status()

        for evento, texto in iterar_sse(response):
            if time.time() - inicio >= duracao_s:
                return

            if evento in ("cancel", "auth_revoked"):
                raise RuntimeError(f"Firebase encerrou o stream: {evento}")

            if evento not in ("put", "patch"):
                continue

            envelope = json.loads(texto)
            data = envelope.get("data")

            if isinstance(data, dict) and "seq" in data:
                yield data


# Função já utilizada anteriormente: valida os campos e converte os sinais para NumPy.
def validar_pacote(pacote):
    campos = (
        "session_id", "seq", "sample_start", "fs", "n",
        "ecg", "ppg_ir", "ppg_red"
    )

    faltando = [k for k in campos if k not in pacote]
    if faltando:
        raise ValueError(f"campos ausentes: {faltando}")

    n = int(pacote["n"])
    ecg = np.asarray(pacote["ecg"], dtype=float)
    ir = np.asarray(pacote["ppg_ir"], dtype=float)
    red = np.asarray(pacote["ppg_red"], dtype=float)

    if not (len(ecg) == len(ir) == len(red) == n):
        raise ValueError("vetores com tamanhos incompatíveis")

    return {
        "session_id": str(pacote["session_id"]),
        "seq": int(pacote["seq"]),
        "sample_start": int(pacote["sample_start"]),
        "fs": float(pacote["fs"]),
        "n": n,
        "timestamp_pub": float(pacote.get("timestamp_pub", 0.0)),
        "ecg": ecg,
        "ppg_ir": ir,
        "ppg_red": red,
    }


## 3. Capturar uma janela contínua

Para aplicar filtros e detectar picos, este desafio precisa de um trecho **sem lacunas**.
Por isso, se houver mudança de sessão ou perda de amostras, o acumulador é reiniciado.

Essa decisão é específica do processamento deste checkpoint; nos checkpoints anteriores,
as lacunas eram preservadas com `NaN`.


In [ ]:
# Captura uma janela contínua reutilizando pacotes_firebase() e validar_pacote().
def capturar_janela_continua(duracao_s=DURACAO_SEGUNDOS):
    ecg = []
    ir = []
    red = []

    fs = None
    active_session = None
    last_sample_end = None

    for pacote in pacotes_firebase(
        firebase_url=FIREBASE_URL,
        node=STREAM_PATH,
        duracao_s=duracao_s,
    ):
        try:
            p = validar_pacote(pacote)
        except ValueError as exc:
            print("Pacote ignorado:", exc)
            continue

        # Uma nova sessão significa um novo sinal; não misturamos sessões diferentes.
        if active_session != p["session_id"]:
            active_session = p["session_id"]
            last_sample_end = None
            ecg.clear()
            ir.clear()
            red.clear()

        # Pacotes duplicados ou antigos não são acrescentados novamente.
        if last_sample_end is not None and p["sample_start"] <= last_sample_end:
            continue

        # O filtro deste checkpoint precisa de uma janela contínua.
        # Se houver uma lacuna, descartamos o trecho anterior e começamos uma nova janela.
        if last_sample_end is not None:
            esperado = last_sample_end + 1
            if p["sample_start"] > esperado:
                print("Lacuna detectada: reiniciando a janela do desafio.")
                ecg.clear()
                ir.clear()
                red.clear()

        fs = p["fs"]

        # extend() acrescenta todas as amostras do pacote à lista existente.
        ecg.extend(p["ecg"])
        ir.extend(p["ppg_ir"])
        red.extend(p["ppg_red"])

        last_sample_end = p["sample_start"] + p["n"] - 1

    if fs is None or len(ecg) == 0:
        raise RuntimeError("Nenhum sinal válido foi recebido. Verifique o publicador e o Firebase.")

    return (
        np.asarray(ecg, dtype=float),
        np.asarray(ir, dtype=float),
        np.asarray(red, dtype=float),
        float(fs),
    )


ec, ir, red, fs = capturar_janela_continua()

print(f"capturadas {len(ec)} amostras")
print(f"fs = {fs:g} Hz")
print(f"duração útil = {len(ec) / fs:.1f} s")


## 4. Filtragem e detecção de picos

Usaremos filtros e detectores simples apenas para demonstrar uma cadeia básica de processamento.


In [ ]:
# Aplica um filtro passa-faixa Butterworth usando filtragem nos dois sentidos.
# sosfiltfilt evita introduzir um deslocamento temporal dos picos no exemplo didático.
def bandpass(x, fs, f1, f2, ordem=3):
    sos = butter(
        ordem,
        [f1, f2],
        btype="bandpass",
        fs=fs,
        output="sos",
    )
    return sosfiltfilt(sos, x)


# Garante que temos dados suficientes para filtrar e detectar mais de um batimento.
if len(ec) < int(3 * fs):
    raise RuntimeError("Janela muito curta. Capture pelo menos alguns segundos de sinal contínuo.")

# ECG: mantém aproximadamente 0,5–40 Hz e procura os picos R.
ecg_f = bandpass(ec, fs, 0.5, 40)
rpeaks, _ = find_peaks(
    ecg_f,
    distance=int(0.35 * fs),  # Impede detectar dois picos cardíacos muito próximos.
    prominence=0.25,          # Exige que o pico se destaque do sinal ao redor.
)

# Calcula a frequência cardíaca pela mediana dos intervalos R-R.
fc = (
    60 / np.median(np.diff(rpeaks) / fs)
    if len(rpeaks) > 1
    else np.nan
)

# PPG: mantém aproximadamente 0,5–5 Hz e procura os pulsos.
ppg_f = bandpass(ir, fs, 0.5, 5)
ppg_peaks, _ = find_peaks(
    ppg_f,
    distance=int(0.35 * fs),
    prominence=0.5 * np.std(ppg_f),  # Limiar adaptado à amplitude do PPG recebido.
)

# Calcula a frequência de pulso pela mediana dos intervalos entre pulsos consecutivos.
fp = (
    60 / np.median(np.diff(ppg_peaks) / fs)
    if len(ppg_peaks) > 1
    else np.nan
)

print(f"FC pelo ECG: {fc:.1f} bpm")
print(f"Frequência de pulso pelo PPG: {fp:.1f} bpm")


## 5. Visualização do resultado


In [ ]:
t = np.arange(len(ec)) / fs  # Reconstrói o eixo de tempo a partir da frequência de amostragem.

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax[0].plot(t, ecg_f, label="ECG filtrado")
ax[0].plot(t[rpeaks], ecg_f[rpeaks], "o", label="picos R")
ax[0].set_ylabel("ECG filtrado")
ax[0].grid(True, alpha=0.25)
ax[0].legend()

ax[1].plot(t, ppg_f, label="PPG IR filtrado")
ax[1].plot(t[ppg_peaks], ppg_f[ppg_peaks], "o", label="pulsos")
ax[1].set_ylabel("PPG IR filtrado")
ax[1].set_xlabel("tempo (s)")
ax[1].grid(True, alpha=0.25)
ax[1].legend()

plt.tight_layout()
plt.show()


## 6. Desafio

1. Compare a frequência cardíaca estimada pelo ECG com a frequência de pulso estimada pelo PPG.
2. Mude os parâmetros de `prominence` e observe falsos positivos/falsos negativos.
3. Repita o processamento usando `ppg_red` no lugar de `ppg_ir`.
4. **Bônus:** com o gerador em `NOISE` ou `MOTION`, observe como a detecção de picos pode se degradar.
5. **Bônus:** meça o atraso entre um pico R e o pulso PPG seguinte, tratando-o apenas como medida temporal didática — não como diagnóstico.

> Antes de iniciar a captura principal deste checkpoint, deixe o gerador em **`NORMAL`**.
